In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
#warnings 제거 및 한글 폰트 추가 가능하게
warnings.filterwarnings('ignore')
plt.rc('font', family='Malgun Gothic')

In [13]:
pd.set_option('display.max_columns', None)

In [14]:
df = pd.read_csv('EternalReturn_kakaogames_2024.csv')

In [15]:
# 사망 관련 컬럼 리스트 (아오 많아)
death_log_cols = [
    'killer', 'killerCharacter', 'killerWeapon', 'causeOfDeath', 'placeOfDeath', 'killDetail',
    'killer2', 'killerCharacter2', 'killerWeapon2', 'causeOfDeath2', 'placeOfDeath2', 'killDetail2',
    'killer3', 'killerCharacter3', 'killerWeapon3', 'causeOfDeath3', 'placeOfDeath3', 'killDetail3'
]

# 결측치를 'None'으로 라벨링
for col in death_log_cols:
    if col in df.columns:
        df[col] = df[col].fillna('None')

In [16]:

# 조언 받아 'user1369612'으로 대체
df['nickname'] = df['nickname'].fillna('user1369612')

In [17]:
# language 필요할 것 같지 않으니 전체 열 삭제
df.drop(columns=['language'], inplace=True)

In [18]:
# 중복 데이터 제거
# 한 게임에 동일 유저가 두 명 들어가면 안 됨.
# userNum으로 해서 상관 없긴 한데, 이터널리턴은 다행히 중복닉 안 된다고 하네요
# gameId와 userNum의 조합 유니크하게
df = df.drop_duplicates(subset=['gameId', 'userNum'])

In [19]:
#게임이 랭크게임인지 판단
game_rank_flag = (
    df
    .groupby("gameId")
    .apply(
        lambda x: (
            (x["mmrGainInGame"] != 0).any() or
            (x["mmrLossEntryCost"] != 0).any()
        )
    )
    .reset_index(name="is_rank_game")
)

# 통째로 df1에 merge
df = df.merge(game_rank_flag, on="gameId", how="left")

# 랭크게임 여부를 확인하는 rank_type_v2 컬럼 추가
df["rank_type"] = df["is_rank_game"].map(
    {True: "rank_game", False: "rank_none"}
)

df["rank_type"].value_counts()

rank_type
rank_none    117995
rank_game     86430
Name: count, dtype: int64

In [20]:
#레벨 별로 나누어 랭크게임 가능/불가능 여부 판별

df["account_level_group"] = df["accountLevel"].apply(
    lambda x: "over 30" if x >= 30 else "under 30"
)


normal_game_df = df[df["rank_type"] == "rank_none"]

normal_game_count = (
    normal_game_df
    .groupby("account_level_group")
    ["gameId"]
    .nunique()
    .reset_index(name="normal_game_count")
)

normal_game_count

,account_level_group,normal_game_count
0,over 30,6167
1,under 30,5181


In [21]:
# 실력별 세그먼트 분류 위해 0점, 1~2000점, 
# 2001~4000점, 4001~6000점, 6001+로 5개의 세그먼트로 분류
# (이상치/결측치 없음, 분류 기준은 추후 변경 가능)
bins = [-1, 2400, 3600, 6400,np.inf]
labels = ['아이언&브론즈', '실버&골드', '플래티넘&다이아', '메테오라이트+']

df['rank_group'] = pd.cut(
    df['rankPoint'],
    bins=bins,
    labels=labels
    )


In [22]:
#캐릭별 status 매칭
import json

with open("character_stat.json", "r", encoding="utf-8") as f:
    j = json.load(f)

df_char = pd.DataFrame(j["data"])                 # code, , maxHp, attackPower, ...
df_char = df_char.rename(columns={"code": "characterNum", "name": "characterName_en"})

df["characterNum"] = pd.to_numeric(df["characterNum"], errors="coerce").astype("Int64")
df_char["characterNum"] = pd.to_numeric(df_char["characterNum"], errors="coerce").astype("Int64")

df = df.merge(df_char, on="characterNum", how="left")

In [23]:
#한글이름 매핑
charactor_n = {1: '재키',
2: '아야',
3:'피오라',
4:'매그너스',
5: '자히르',
6:'나딘',
7:'현우',
8:'하트',
9:'아이솔',
10:'리 다이린',
11:'유키',
12:'혜진',
13:'쇼우',
14:'키아라',
15:'시셀라',
 16:'실비아',
17:'아드리아나',
 18:'쇼이치',
 19:'엠마',
 20:'레녹스',
 21:'로지',
 22:'루크',
 23:'캐시',
 24:'아델라',
 25:'버니스',
 26:'바바라',
 27:'알렉스',
 28:'수아',
 29:'레온',
 30:'일레븐',
 31:'리오',
 32:'윌리엄',
 33:'니키',
 34:'나타폰',
 35:'얀',
 36:'이바',
 37:'다니엘',
 38:'제니',
 39:'카밀로',
 40:'클로에',
 41:'요한',
 42:'비앙카',
 43:'셀린',
 44:'에키온',
 45:'마이',
 46:'에이든',
 47:'라우라',
 48:'띠아',
 49:'펠릭스',
 50:'엘레나',
 51:'프리야',
 52:'아디나',
53:'마커스',
54:'칼라',
55:'에스텔',
56:'피올로',
57:'마르티나',
58:'헤이즈',
59:'아이작',
60:'타지아',
61:'이렘',
62:'테오도르',
63:'이안',
64:'바냐',
65:'데비&마를렌',
66:'아르다',
67:'아비게일',
68:'알론소',
69:'레니',
70:'츠바메',
71:'케네스',
72:'카티야',
73:'샬럿',
74:'다르코',
75:'르노어',
76:'가넷',
77:'유민',
78:'히스이',
79:'유스티나',
80:'이슈트반',
81:'니아',82:'슈린',83:'헨리',84:'블레어',85:'미르카',9999:'나쟈'}
df['characterName_kr'] = df['characterNum']
df['characterName_kr'] = df['characterName_kr'].map(charactor_n)

유저 대시보드 데이터 추출

In [25]:
# 유저 대시보드 데이터 추출
# 팀장님 감사합니다

required_columns = [
    # 기본 식별 및 세그먼트
    'accountLevel', 'gameId', 'userNum', 'serverName', 'gameRank', 'rankPoint', 'matchingTeamMode', 
    'versionMajor', 'versionMinor',
    
    # 6대 KPI 계산용 (승률, KDA, DPM, 생존시간)
    'victory', 'playerKill', 'playerAssistant', 'playerDeaths', 
    'damageToPlayer', 'damageFromPlayer', 'playTime', 'survivableTime',
    
    # 팀 서포트 및 다양성 (회복, 보호막, 시야, 캐릭터)
    'teamRecover', 'protectAbsorb', 'viewContribution', 'characterNum',
    
    # 자원 전환 및 성장 효율
    'totalGainVFCredit', 'totalUseVFCredit',  # 총 획득/사용 크레딧
    'transferConsoleFromRevivalUseVFCredit', # 부활에 쓴 크레딧(안 필요할 수도 근데 그냥 제 경험 기반으로 넣어놓음)
    'transferConsoleFromMaterialUseVFCredit', # 재료에 쓴 크레딧
    'craftEpic', 'craftLegend', 'craftMythic', # 제작 아이템 등급별 수량
    
    # 개인 추세 및 MMR
    'mmrGainInGame', 'mmrLossEntryCost',
    
    #
    'rank_group','isLeavingBeforeCreditRevivalTerminate', 'giveUp', "account_level_group", "rank_type"

]

# 더 필요한 컬럼이 있다면 추후 추가하면 됨
# 태블로에서 데이터-새로고침 하면 됩니다

core_df = df[required_columns].copy()

core_df

,accountLevel,gameId,userNum,serverName,gameRank,rankPoint,matchingTeamMode,versionMajor,versionMinor,victory,playerKill,playerAssistant,playerDeaths,damageToPlayer,damageFromPlayer,playTime,survivableTime,teamRecover,protectAbsorb,viewContribution,characterNum,totalGainVFCredit,totalUseVFCredit,transferConsoleFromRevivalUseVFCredit,transferConsoleFromMaterialUseVFCredit,craftEpic,craftLegend,craftMythic,mmrGainInGame,mmrLossEntryCost,rank_group,isLeavingBeforeCreditRevivalTerminate,giveUp,account_level_group,rank_type
0,119,35260427,4378745,Seoul,5,5634,3,22,1,0,6,1,3,15206,14873,1125,23,0,3767,21,49,1209,1082,22,800,7,6,0,30,-44,플래티넘&다이아,False,0,over 30,rank_game
1,82,35260427,4313007,Seoul,4,5625,3,22,1,0,2,6,2,11130,16891,1220,23,0,2534,27,28,1265,998,168,600,6,5,0,69,-44,플래티넘&다이아,False,0,over 30,rank_game
2,133,35260427,4227659,Seoul,2,5285,3,22,1,0,2,8,3,16795,15347,1145,12,988,5744,18,23,1068,780,0,370,8,5,0,128,-43,플래티넘&다이아,False,0,over 30,rank_game
3,156,35260427,3979035,Seoul,1,5316,3,22,1,1,4,3,0,12282,15312,1342,60,0,3238,31,53,1302,1130,200,400,5,4,0,124,-43,플래티넘&다이아,False,0,over 30,rank_game
4,150,35260427,2118488,Seoul,1,5439,3,22,1,1,2,4,2,11339,15478,1342,60,0,0,25,65,1186,1120,0,650,7,6,0,124,-43,플래티넘&다이아,False,0,over 30,rank_game
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204420,10,35765219,3927798,Europe,3,0,3,23,0,0,0,0,1,0,0,838,0,0,0,0,68,464,0,0,0,0,0,0,0,0,아이언&브론즈,True,1,under 30,rank_none
204421,16,35783095,4932642,Asia,7,0,3,23,0,0,0,0,1,0,0,147,20,0,0,0,36,116,0,0,0,0,0,0,0,0,아이언&브론즈,True,1,under 30,rank_none
204422,118,35796912,456574,Asia,8,0,3,23,0,0,0,0,1,0,0,90,20,0,0,0,74,55,10,0,0,0,0,0,0,0,아이언&브론즈,True,1,over 30,rank_none
204423,56,35804415,4519640,Asia,8,4368,3,23,0,0,0,0,1,0,0,48,20,0,0,0,8,42,0,0,0,0,0,0,0,-72,플래티넘&다이아,True,1,over 30,rank_game


In [26]:
# 수치 통일 위해 정규화
core_df['scoreRecover'] = df['teamRecover'] / df['teamRecover'].replace(0, 1).max()
core_df['scoreProtect'] = df['protectAbsorb'] / df['protectAbsorb'].replace(0, 1).max()
core_df['scoreVision'] = df['viewContribution'] / df['viewContribution'].replace(0, 1).max()
core_df['scoreCreditRevive'] = df['creditRevivedOthersCount'] / df['creditRevivedOthersCount'].replace(0, 1).max()

In [27]:
#core_df 최종 저장 
core_df.to_csv('ER_core_data.csv', index=False, encoding='utf-8-sig')

콘텐츠 대시보드 데이터 추출

In [30]:
colunms_content = [
'characterName_kr',
'gameId'	,
'characterNum',	
'bestWeapon',
'gameRank',
'playerKill',	
'playerAssistant',	
'playerDeaths'	,
#'equipment'	,
'totalGainVFCredit'  ,
'killPlayerGainVFCredit',
'killChickenGainVFCredit',
'killBoarGainVFCredit',
'killWildDogGainVFCredit',
'killWolfGainVFCredit',
'killBearGainVFCredit',
'killOmegaGainVFCredit',
'killBatGainVFCredit',
'killWicklineGainVFCredit',
'killAlphaGainVFCredit',
'killItemBountyGainVFCredit',
'killDroneGainVFCredit',
'totalUseVFCredit'     ,
'remoteDroneUseVFCreditMySelf',
'remoteDroneUseVFCreditAlly',
'transferConsoleFromMaterialUseVFCredit',
'transferConsoleFromEscapeKeyUseVFCredit',
'transferConsoleFromRevivalUseVFCredit',
'tacticalSkillUpgradeUseVFCredit',
'viewContribution',
'maxHp_y',
'attackPower_y',
'defense_y',
'attackSpeed_y',
'attackRange'
]

In [31]:
#사용할 컬럼만
df_teab = df[colunms_content]

In [32]:
# 수치 통일 위해 정규화
df_teab['scoreRecover'] = df['teamRecover'] / df['teamRecover'].replace(0, 1).max()
df_teab['scoreProtect'] = df['protectAbsorb'] / df['protectAbsorb'].replace(0, 1).max()
df_teab['scoreVision'] = df['viewContribution'] / df['viewContribution'].replace(0, 1).max()
df_teab['scoreCreditRevive'] = df['creditRevivedOthersCount'] / df['creditRevivedOthersCount'].replace(0, 1).max()

In [33]:
df_teab.to_csv('컨텐츠대시보드_데이터.csv', index=False,encoding='UTF-8')